In [2]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import r2_score

import pn.CNN3D as CNN3D
import pn.principal_component_analysis as pca_helper
import importlib
importlib.reload(pca_helper)
import pn.p5_utils as p5u
from sklearn.manifold import TSNE
from sklearn.cross_decomposition import CCA

In [6]:
importlib.reload(pca_helper)
pcas = {}
original_sessions = {}
target_patient = 2
all_patients = [2, 3, 4]
non_target_patients = [3, 4]

for patient in all_patients:
    config = CNN3D.Config(patient)
    original_sessions[patient] = pca_helper.get_recordings(config, config.recordings, CNN3D.Config(target_patient).smooth_train_ms)

Using default channels, all inbound electrodes will be included.
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Training block 1_green__loc4_7_1742409770184_250319_144250_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Training block 2_green__loc8_1_1742410047212_250319_144727_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Training block 3_green__loc4_3_1742410337516_250319_145217_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Validation block_green_loc8_2_1742410614837_250319_145654_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preproces

In [7]:
importlib.reload(pca_helper)
for patient in all_patients:
    pca, transformed_data = pca_helper.get_pca(patient, original_sessions[patient] if patient != target_patient else original_sessions[patient][:-2])
    pcas[patient] = (pca, transformed_data)

torch.Size([33, 33, 25085])
(75498, 1089)
(90, 1089) components for patient 2, total variance explained: 92.490341228306%
torch.Size([33, 33, 24784])
(125339, 1089)
(90, 1089) components for patient 3, total variance explained: 88.44095790020421%
torch.Size([33, 33, 25597])
(134585, 1089)
(90, 1089) components for patient 4, total variance explained: 84.08719295841259%


In [9]:
pca_t = pcas[target_patient][1]
print(pca_t[target_patient].shape)
pca_t_all_sessions = np.concat([session for session in pca_t])
ccas = {}
for patient in [3, 4]:
    # Get the transformed latent space data for the source and target patients
    pca_s = pcas[patient][1]
    pca_s_all_sessions = np.concat([session for session in pca_s])

    cca = CCA(n_components=90)
    min_length = min(len(pca_t_all_sessions), len(pca_s_all_sessions))
    cca.fit(pca_s_all_sessions[:min_length], pca_t_all_sessions[:min_length])

    print(f"The shape of the target patient's PCA is {len(pca_t_all_sessions)} by {len(pca_t_all_sessions[0])}, the shape of patient {patient}'s is {len(pca_s_all_sessions)} by {len(pca_s_all_sessions[0])}")

    pca_s_c = [cca.transform(session) for session in pca_s]
    print(f"The shape Patient {patient}'s CCA transformed data is {len(pca_s_c)} by {len(pca_s_c[0])} by {len(pca_s_c[0][0])}")
    ccas[patient] = (cca, pca_s_c)


(25472, 90)


KeyboardInterrupt: 

In [ ]:
print(len(pcas[target_patient][1][0]))

# pca_transform = pcas[0][0].transform(pcas[0][1])
# print(len(pca_transform[0]))

aligned_recordings = {}

for patient in non_target_patients:
    _, latent_recordings = ccas[patient]
    aligned_recordings[patient] = []
    for latent_recording in latent_recordings:

        transformed_recording = pcas[target_patient][0].inverse_transform(latent_recording)
        reshaped_recording = np.transpose(np.reshape(transformed_recording, (transformed_recording.shape[0], 33, 33)), (1, 2, 0))

        aligned_recordings[patient].append(torch.tensor(reshaped_recording, dtype=torch.float32))

# aligned_target_data = pcas[target_patient][0].inverse_transform(ccas[target_patient][1][0])

print(f"Aligned target data is of shape {len(aligned_recordings)} by {len(aligned_recordings[3])} by {aligned_recordings[3][0].shape}")

# ccas[0][0].transform(pca_transform, pcas[1][0].components_)

In [ ]:
config = CNN3D.Config(target_patient)
# val_sessions = pca_helper.get_recordings(config, [config.recordings[-2]], config.smooth_val_ms)
# test_sessions = pca_helper.get_recordings(config, [config.recordings[-1]], config.smooth_val_ms)
val_sessions = [original_sessions[target_patient][-2]]
test_sessions = [original_sessions[target_patient][-1]]


In [ ]:
importlib.reload(pca_helper)
importlib.reload(CNN3D)

# print(original_sessions[2][2][2].shape)
train_loader, valid_loader, test_loader = pca_helper.get_aligned_data_loader_v2(target_patient, aligned_recordings, original_sessions, val_sessions, test_sessions)

In [ ]:
importlib.reload(CNN3D)
config = CNN3D.Config(target_patient)
numIter = 20

p1_r2 = []
p1_pmse = []
for rep in range(numIter):
    print("------------------------")
    print(f'Iter {rep+1}')
    # make model
    model = CNN3D.CNN2D_LSTM(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = torch.nn.MSELoss(reduction='sum')

    # train
    CNN3D.CNN3D_train(model, optimizer, criterion, config, train_loader, valid_loader)

    tgts2, preds2 = CNN3D.CNN3D_predict(model, test_loader)

    p1_r2.append(r2_score(tgts2, preds2))
    p1_pmse.append(np.mean(np.linalg.norm(tgts2 - preds2,axis=1)**2)/2)
    print("------------------------")

In [3]:
train_loader, valid_loader, test_loader = pca_helper.get_aligned_data_loader(2, [2, 3, 4])

Using default channels, all inbound electrodes will be included.
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Training block 1_green__loc4_7_1742409770184_250319_144250_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Training block 2_green__loc8_1_1742410047212_250319_144727_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Training block 3_green__loc4_3_1742410337516_250319_145217_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preprocessed-features-V2/NSR-004-002/Validation block_green_loc8_2_1742410614837_250319_145654_hga.pt
Smoothing
/Users/orhanelam/Library/CloudStorage/OneDrive2-SharedLibraries-JohnsHopkins/cronelab - PN/JHU-preproces

In [5]:
importlib.reload(CNN3D)
config = CNN3D.Config(2)
numIter = 20

p1_r2 = []
p1_pmse = []
for rep in range(numIter):
    print("------------------------")
    print(f'Iter {rep+1}')
    # make model
    model = CNN3D.CNN2D_LSTM(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = torch.nn.MSELoss(reduction='sum')

    # train
    CNN3D.CNN3D_train(model, optimizer, criterion, config, train_loader, valid_loader)

    tgts2, preds2 = CNN3D.CNN3D_predict(model, test_loader)

    p1_r2.append(r2_score(tgts2, preds2))
    p1_pmse.append(np.mean(np.linalg.norm(tgts2 - preds2,axis=1)**2)/2)
    print("------------------------")

------------------------
Iter 1
Epoch 0/30: train:8.344548882664862  val:15.805968093872071


KeyboardInterrupt: 

In [ ]:
importlib.reload(pca_helper)
train_loader, valid_loader, test_loader = pca_helper.get_pca_data_loader(2, records)

In [ ]:
importlib.reload(CNN3D)
config = CNN3D.Config(2)
numIter = 30
p1_r2 = []
p1_pmse = []
for rep in range(numIter):
    print("------------------------")
    print(f'Iter {rep + 1}')
    # make model
    model = CNN3D.CNN2D_LSTM(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = torch.nn.MSELoss(reduction='sum')

    # train
    CNN3D.CNN3D_train(model, optimizer, criterion, config, train_loader, valid_loader)

    tgts2, preds2 = CNN3D.CNN3D_predict(model, test_loader)

    p1_r2.append(r2_score(tgts2, preds2))
    p1_pmse.append(np.mean(np.linalg.norm(tgts2 - preds2, axis=1) ** 2) / 2)
    print("------------------------")